## Scratchpad

In [1]:
from lcn.core.model import LCN

# Load an example LCN
filename = "/Users/radu/git/IBM/LCN/benchmarks/chain/chain_n20_1.lcn"

lcn = LCN()
lcn.from_lcn(file_name=filename)
lcn.summary()

G1 = lcn.build_primal_graph(formula_labels=True)
G2 = lcn.build_structure_graph()

Parsed LCN format with 25 sentences.
Build the LCN's primal graph.
Build the LCN's structure graph.
Processing sentence: s0: 0.327505 <= P(x14) <= 0.927505 ; True
Processing sentence: s1: 0.03637 <= P((x18 xor !x15)) <= 0.63637 ; True
Processing sentence: s2: 0.054622 <= P((!x0 and !x2)) <= 0.654622 ; True
Processing sentence: s3: 0.0 <= P((!x7 or !x4)) <= 0.317874 ; True
Processing sentence: s4: 0.0 <= P((!x10 and x5)) <= 0.424438 ; True
Processing sentence: s5: 0.313411 <= P((x16 or (x19 xor x11))) <= 0.913411 ; True
Processing sentence: s6: 0.0 <= P(x3) <= 0.531672 ; True
Processing sentence: s7: 0.349647 <= P(x13) <= 0.949647 ; True
Processing sentence: s8: 0.212689 <= P(x8) <= 0.812689 ; True
Processing sentence: s9: 0.0 <= P(!x1) <= 0.476528 ; True
Processing sentence: s10: 0.0 <= P(!x17) <= 0.486438 ; True
Processing sentence: s11: 0.050125 <= P(x9) <= 0.650125 ; True
Processing sentence: s12: 0.583786 <= P(!x14 | x13) <= 1.0 ; True
Processing sentence: s13: 0.191186 <= P(x8 | (

In [ ]:
%matplotlib inline
from lcn.core.plotting import plot_graph

plot_graph(G1, layout="kamada_kawai", figsize=(10, 8), title="Primal graph");

In [ ]:
plot_graph(G2, label_attr="label", title="Structure graph");

In [4]:
# check if the LCN is a chain graph 
ok = lcn.is_chain_graph()
print(f"Is the LCN a chain graph? {ok}")

Simplifying the structure by replacing the undirected cliques
Is the LCN a chain graph? True


In [5]:
# get the families of each node in the chain graph
families = lcn.process_chain_graph()
print("Families of each node:")
for family in families:
    node = family["child"]
    print(f"{node}: {family}")

Child: x1, Parents: ['x12'], Scope: ['x1', 'x12']
Adding sentence s9 with scope {'x1'}.
Adding sentence s15 with scope {'x12', 'x1'}.
Child: x12, Parents: [], Scope: ['x12']
Child: x13, Parents: [], Scope: ['x13']
Adding sentence s7 with scope {'x13'}.
Child: x14, Parents: ['x13'], Scope: ['x14', 'x13']
Adding sentence s0 with scope {'x14'}.
Adding sentence s7 with scope {'x13'}.
Adding sentence s12 with scope {'x13', 'x14'}.
Child: x17, Parents: ['x8'], Scope: ['x17', 'x8']
Adding sentence s8 with scope {'x8'}.
Adding sentence s10 with scope {'x17'}.
Adding sentence s16 with scope {'x17', 'x8'}.
Child: x3, Parents: [], Scope: ['x3']
Adding sentence s6 with scope {'x3'}.
Child: x6, Parents: ['x8'], Scope: ['x6', 'x8']
Adding sentence s8 with scope {'x8'}.
Adding sentence s22 with scope {'x8', 'x6'}.
Child: x8, Parents: ['x12', 'x14'], Scope: ['x8', 'x12', 'x14']
Adding sentence s0 with scope {'x14'}.
Adding sentence s8 with scope {'x8'}.
Adding sentence s13 with scope {'x8', 'x14', 'x1

In [ ]:
# Build the symbolic chain-graph factorization (one factor P(child | parents)
# per family). The class moved to lcn.inference.marginal.cn and was renamed.
from lcn.inference.marginal.cn.factorization import ChainGraphFactorization

fact = ChainGraphFactorization(lcn)
factors = fact.build()

print("Symbolic chain-graph factors P(child | parents):")
for f in factors:
    print(f"  P({f['child']} | {f['parents']})  scope={f['scope']}  "
          f"sentences={f['sentences']}")


In [ ]:
# Compile the LCN to a credal network and enumerate the extreme points of each
# local credal set. `method`: "linear" (in-scope-sentence LP, default) or
# "linear-tight" (adds the scope-restricted LMC equalities -- scheme D1).
# `solver`: "ipopt" (local, default) or "scip" (certified global -- scheme D3).
# Set verbosity=2 to stream the ipopt/scip solver progress.
from lcn.inference.marginal.cn.vertices import CredalNetworkVertices

cnv = CredalNetworkVertices.from_lcn(lcn, method="linear", verbosity=1)


In [ ]:
# Credal Variable Elimination now computes ALL posterior marginals in one run():
# it loops the per-target bucket elimination over the network nodes and returns
# {name -> (lower, upper)}. It also records the running times and flags a
# DEGENERATE (all-vacuous [0,1]) solution.
from lcn.inference.marginal.cn.cve import CredalVE

cve = CredalVE(cnv=cnv)
results = cve.run(evidence={}, verbosity=1)

print("\nSingleton-atom marginals P(atom=1):")
for atom in sorted(cve.singleton_marginals):
    lo, hi = cve.singleton_marginals[atom]
    print(f"  P({atom}=1) in [{lo:.6f}, {hi:.6f}]")

print(f"\nbuild time={cve.build_time:.4f}s, "
      f"elimination time={cve.elimination_time:.4f}s, "
      f"total time={cve.total_time:.4f}s")
print(f"degenerate (all marginals vacuous [0,1])? {cve.degenerate}")


In [ ]:
# Tighter / exact options (schemes D4 and D5), opt-in via `coupling`:
#   "off"          -- bounds the strong extension (default, above);
#   "cross-family" -- D4, forbids vertex combinations that violate a
#                     cross-family LCN sentence/LMC assertion;
#   "d5"           -- D5, the junction-tree EXACT NLP (defaults to the certified
#                     global SCIP backend). Exact for both marginal and
#                     conditional queries.
cve_d5 = CredalVE(cnv=cnv)
cve_d5.run(evidence={}, coupling="d5", verbosity=1)

print("\nExact marginals via D5:")
for atom in sorted(cve_d5.singleton_marginals):
    lo, hi = cve_d5.singleton_marginals[atom]
    print(f"  P({atom}=1) in [{lo:.6f}, {hi:.6f}]")
